In [87]:
import pandas as pd

In [88]:
#Notice: there are missing values in both Unit_Price and Cost_Price, and Transaction T021 has a Store_ID of S07 which does not exist in the stores file.
# Import both CSVs. Convert date columns to datetime. Check for missing values in the sales data and report the count per column. 
# Fill missing Unit_Price values with the average Unit_Price for that specific Product (group-specific fill). 
# Fill missing Cost_Price values with the median Cost_Price for that specific Category. Verify no nulls remain.

rsales = pd.read_csv('retail_sales.csv')
rsales['Sale_Date'] = pd.to_datetime(rsales['Sale_Date'])
rsales.isnull().sum()
rsales['Unit_Price'] = rsales['Unit_Price'].fillna(rsales.groupby('Product')['Unit_Price'].transform('mean'))
rsales['Cost_Price'] = rsales['Cost_Price'].fillna(rsales.groupby('Category')['Cost_Price'].transform('median'))
rsales.isnull().sum()
rsales

,Transaction_ID,Store_ID,Sale_Date,Product,Category,Quantity,Unit_Price,Cost_Price,Customer_ID
0,T001,S01,2024-01-05,Jacket,Apparel,2,89.99,45.0,CU01
1,T002,S01,2024-01-08,Sneakers,Footwear,1,129.99,65.0,CU02
2,T003,S02,2024-01-10,Jacket,Apparel,1,89.99,45.0,CU03
3,T004,S03,2024-01-12,Boots,Footwear,2,149.99,65.0,CU01
4,T005,S01,2024-01-15,Hat,Accessories,3,24.99,10.0,CU04
5,T006,S03,2024-01-18,Sneakers,Footwear,1,129.99,65.0,CU05
6,T007,S04,2024-01-20,Jacket,Apparel,2,89.99,45.0,CU02
7,T008,S02,2024-01-22,Hat,Accessories,5,24.99,10.0,CU06
8,T009,S05,2024-01-25,Boots,Footwear,1,149.99,75.0,CU03
9,T010,S06,2024-01-28,Sneakers,Footwear,2,129.99,65.0,CU07


In [89]:
#Notice: there are missing values in both Unit_Price and Cost_Price, and Transaction T021 has a Store_ID of S07 which does not exist in the stores file.
# Import both CSVs. Convert date columns to datetime. Check for missing values in the sales data and report the count per column. 
# Fill missing Unit_Price values with the average Unit_Price for that specific Product (group-specific fill). 
# Fill missing Cost_Price values with the median Cost_Price for that specific Category. Verify no nulls remain.

rstores = pd.read_csv('retail_stores.csv')
rstores['Open_Date'] = pd.to_datetime(rstores['Open_Date'])
rstores.isnull().sum()
rstores

,Store_ID,Store_Name,Region,Manager,Open_Date
0,S01,Downtown,East,Maria,2020-03-15
1,S02,Westside,West,James,2021-07-01
2,S03,Northgate,North,Maria,2019-11-20
3,S04,Southpark,South,James,2022-01-10
4,S05,Eastview,East,Linda,2023-06-01
5,S06,Central,North,Linda,2018-08-25


In [90]:
# Left join the sales data with the store data on Store_ID. Identify any transactions that did not match a store (the S07 issue). 
# Create a column called Store_Known — True if the store matched, False if not. Keep all transactions.

result = rsales.merge(rstores, on = 'Store_ID', how = 'left')
result['Store_Known'] = result['Store_Name'].notna()
result

,Transaction_ID,Store_ID,Sale_Date,Product,Category,Quantity,Unit_Price,Cost_Price,Customer_ID,Store_Name,Region,Manager,Open_Date,Store_Known
0,T001,S01,2024-01-05,Jacket,Apparel,2,89.99,45.0,CU01,Downtown,East,Maria,2020-03-15,True
1,T002,S01,2024-01-08,Sneakers,Footwear,1,129.99,65.0,CU02,Downtown,East,Maria,2020-03-15,True
2,T003,S02,2024-01-10,Jacket,Apparel,1,89.99,45.0,CU03,Westside,West,James,2021-07-01,True
3,T004,S03,2024-01-12,Boots,Footwear,2,149.99,65.0,CU01,Northgate,North,Maria,2019-11-20,True
4,T005,S01,2024-01-15,Hat,Accessories,3,24.99,10.0,CU04,Downtown,East,Maria,2020-03-15,True
5,T006,S03,2024-01-18,Sneakers,Footwear,1,129.99,65.0,CU05,Northgate,North,Maria,2019-11-20,True
6,T007,S04,2024-01-20,Jacket,Apparel,2,89.99,45.0,CU02,Southpark,South,James,2022-01-10,True
7,T008,S02,2024-01-22,Hat,Accessories,5,24.99,10.0,CU06,Westside,West,James,2021-07-01,True
8,T009,S05,2024-01-25,Boots,Footwear,1,149.99,75.0,CU03,Eastview,East,Linda,2023-06-01,True
9,T010,S06,2024-01-28,Sneakers,Footwear,2,129.99,65.0,CU07,Central,North,Linda,2018-08-25,True


In [91]:
# For matched stores only, create columns for Revenue (Quantity * Unit_Price), Cost (Quantity * Cost_Price), and Profit (Revenue - Cost). 
# Then calculate the cumulative Profit per store ordered by Sale_Date. Also calculate the rank of each store by total Profit.

matched = result[result['Store_Known'] == True].copy()
matched['Revenue'] = (matched['Quantity'] * matched['Unit_Price'])
matched['Cost'] = (matched['Quantity'] * matched['Cost_Price'])
matched['Profit'] = (matched['Revenue'] - matched['Cost'])
matched['Cumulative_Profit'] = matched.groupby('Store_ID')['Profit'].cumsum()
store_profit = matched.groupby('Store_ID')['Profit'].sum().reset_index(name='Total_Profit')
store_profit['Rank'] = store_profit['Total_Profit'].rank(ascending=False).astype(int)
store_profit
matched

,Transaction_ID,Store_ID,Sale_Date,Product,Category,Quantity,Unit_Price,Cost_Price,Customer_ID,Store_Name,Region,Manager,Open_Date,Store_Known,Revenue,Cost,Profit,Cumulative_Profit
0,T001,S01,2024-01-05,Jacket,Apparel,2,89.99,45.0,CU01,Downtown,East,Maria,2020-03-15,True,179.98,90.0,89.98,89.98
1,T002,S01,2024-01-08,Sneakers,Footwear,1,129.99,65.0,CU02,Downtown,East,Maria,2020-03-15,True,129.99,65.0,64.99,154.97
2,T003,S02,2024-01-10,Jacket,Apparel,1,89.99,45.0,CU03,Westside,West,James,2021-07-01,True,89.99,45.0,44.99,44.99
3,T004,S03,2024-01-12,Boots,Footwear,2,149.99,65.0,CU01,Northgate,North,Maria,2019-11-20,True,299.98,130.0,169.98,169.98
4,T005,S01,2024-01-15,Hat,Accessories,3,24.99,10.0,CU04,Downtown,East,Maria,2020-03-15,True,74.97,30.0,44.97,199.94
5,T006,S03,2024-01-18,Sneakers,Footwear,1,129.99,65.0,CU05,Northgate,North,Maria,2019-11-20,True,129.99,65.0,64.99,234.97
6,T007,S04,2024-01-20,Jacket,Apparel,2,89.99,45.0,CU02,Southpark,South,James,2022-01-10,True,179.98,90.0,89.98,89.98
7,T008,S02,2024-01-22,Hat,Accessories,5,24.99,10.0,CU06,Westside,West,James,2021-07-01,True,124.95,50.0,74.95,119.94
8,T009,S05,2024-01-25,Boots,Footwear,1,149.99,75.0,CU03,Eastview,East,Linda,2023-06-01,True,149.99,75.0,74.99,74.99
9,T010,S06,2024-01-28,Sneakers,Footwear,2,129.99,65.0,CU07,Central,North,Linda,2018-08-25,True,259.98,130.0,129.98,129.98


In [92]:
# Create a pivot table showing average Profit per transaction by Region (rows) and Category (columns). 
# Use .map() with a dictionary to add a column that labels each Region's performance: 'Strong' if their total Profit > 500, 'Developing' otherwise.

pt = pd.pivot_table(matched, values = 'Profit', index = 'Region', columns = 'Category', aggfunc = 'mean', fill_value = 0).round(2)
region_profit = matched.groupby('Region')['Profit'].sum()
performance_dict = {region: 'Strong' if profit > 500 else 'Developing' for region, profit in region_profit.items()}
pt['Performance'] = pt.index.map(performance_dict)
pt

Category,Accessories,Apparel,Footwear,Performance
Region,,,,
East,44.97,74.98,91.24,Strong
North,44.97,134.97,107.49,Strong
South,89.94,89.98,194.97,Developing
West,74.95,44.99,69.99,Developing


In [93]:
# For each Customer_ID, find their most purchased Product using .drop_duplicates() after sorting. 
# Then identify which customer has the highest total Revenue using .idxmax(). 
# Calculate the month-over-month Revenue growth using .shift(). 
# Finally, use .agg() with named aggregations to show per-Manager total Revenue, average Profit, and number of unique stores — display with a rank column sorted by total Revenue descending.

customer_products = matched.groupby(['Customer_ID', 'Product']).size().reset_index(name='Purchase_Count')
customer_products = customer_products.sort_values('Purchase_Count', ascending=False)
top_product_per_customer = customer_products.drop_duplicates(subset='Customer_ID')
top_product_per_customer

customer_revenue = matched.groupby('Customer_ID')['Revenue'].sum()
customer_revenue
top_customer = customer_revenue.idxmax()
print(f"Customer with highest total Revenue: {top_customer}")

matched['Month'] = matched['Sale_Date'].dt.month
monthly_revenue = matched.groupby('Month')['Revenue'].sum().reset_index(name='Monthly_Revenue')
monthly_revenue['MoM_Growth'] = ((monthly_revenue['Monthly_Revenue'] - monthly_revenue['Monthly_Revenue'].shift(1)) / monthly_revenue['Monthly_Revenue'].shift(1) * 100).round(1)
monthly_revenue

manager_stats = matched.groupby('Manager').agg(Total_Revenue = ('Revenue', 'sum'), Avg_Profit = ('Profit', 'mean'), Unique_Store = ('Store_ID', 'nunique')).reset_index()
manager_stats['Rank'] = manager_stats['Total_Revenue'].rank(ascending=False).astype(int)
manager_stats = manager_stats.sort_values('Rank')
manager_stats

Customer with highest total Revenue: CU01


,Manager,Total_Revenue,Avg_Profit,Unique_Store,Rank
2,Maria,1499.79,79.479000,2,1
0,James,1214.81,90.687143,2,2
1,Linda,1069.90,89.983333,2,3
